# AIエンジニアリング実践講座2026｜第2回・発展宿題用ノートブック

作成日時（日本時間）：**2026/10/28 12:13**  
講義配布の `lora_format_tuning.ipynb`（バージョン1.1）を、**第2回宿題「発展」** の指定に沿って変更した実行用コピー。元ノートブックには変更を加えていない。

**目的**：講座用語集から作った116件のQ&AをLoRAで学習し、既知の質問・言い換え・用語集外について学習前後を比較する。さらに簡易な用語集検索と比較する。

- Google Colab **T4 GPU** を使用する。
- 左のファイルタブから `glossary_sft.jsonl` をアップロードしてから、上から順にセルを実行する。
- **Google Drive連携、APIキー、課金APIは不要**。`/content` のファイルはセッション終了時に消えるので、結果は最後にダウンロードする。
- 実行結果は実測後に確定する。ノートブック中の期待値を結果とみなさない。
- 本課題で用いる学習形式は**通常の会話Q&A**（`messages`）であり、標準演習の架空FAQの `answer / needs_review` JSON形式とは異なる。したがって評価も用語集用に作り直す。


## 0. GPUとライブラリ
講義指定版を使用する。ライブラリの導入後に再起動を要求された場合は、上から実行し直す。

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. 設定とデータ
実験条件を固定する。標準演習は14件×20エポックだが、今回は116件を扱うため**3エポック**を採用し、学習時間と過学習リスクを抑える。比較時の生成条件は共通とする。

In [3]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 3,  # 発展宿題: 用語集116件×3周。標準演習14件×20周とは条件が異なる
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/lec02_advanced",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/lec02_advanced",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # 本課題はGoogle Driveに接続しない
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 3,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 256,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/lec02_advanced",
  "drive_dir": "/content/drive/MyDrive/aie2026/lec02_advanced"
}


In [4]:
from pathlib import Path
import json, re
DATA_PATH = Path("/content/glossary_sft.jsonl")
assert DATA_PATH.exists(), "左のファイル欄から glossary_sft.jsonl をアップロードしてください"
qa = [json.loads(line) for line in DATA_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
assert len(qa) == 116, f"期待116件、実際{len(qa)}件。ファイルを確認してください"
assert all([m['role'] for m in row['messages']] == ['user','assistant'] for row in qa)
questions = {row['messages'][0]['content'] for row in qa}
assert len(questions) == len(qa), "重複質問あり"
print('教師データ', len(qa), '件、先頭の質問：', qa[0]['messages'][0]['content'])
print('先頭の回答：', qa[0]['messages'][1]['content'])
print('Driveへの接続は不要')


教師データ 116 件、先頭の質問： 「AIエンジニアリング」とは何ですか？
先頭の回答： 学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。
Driveへの接続は不要


## 2. 評価データ（学習に使わない質問を含む）
「用語集に掲載された原文質問」6問、「言い換え」6問、「用語集にない項目」6問を事前に固定する。

**採点の限定**：掲載用語は、用語集の定義から選んだ必須語句を回答が含むかという簡易採点である。語句がなくても意味が正しい回答はあり得るため、後で原文を必ず確認する。用語集外は、**用語集への収録有無**であって一般知識の正誤ではない。「未掲載と回答できた率」を別指標にする。

In [5]:
# 評価用の正解語句は116件の用語集定義を確認して事前に作成した。
# same/para は一切の学習前後で同一質問・同一基準を使用。
EVAL_CASES = [
    dict(group="same", q="「トークン」とは何ですか？", keys=["単位"]),
    dict(group="same", q="「LoRA」とは何ですか？", keys=["重み", "行列"]),
    dict(group="same", q="「量子化」とは何ですか？", keys=["重み", "ビット"]),
    dict(group="same", q="「GitHub Codespaces」とは何ですか？", keys=["クラウド", "開発"]),
    dict(group="same", q="「過学習」とは何ですか？", keys=["学習", "未知"]),
    dict(group="same", q="「MCP（Model Context Protocol）」とは何ですか？", keys=["ツール", "共通"]),
    dict(group="para", q="LoRAでは、どの重みを固定し、何を学習するのですか？", keys=["重み", "行列"]),
    dict(group="para", q="モデルの重みを4ビットにしてメモリを節約する技術は？", keys=["量子化", "メモリ"]),
    dict(group="para", q="MCPはAIアプリと外部ツールの連携にどう役立ちますか？", keys=["ツール", "共通"]),
    dict(group="para", q="過学習は未知の入力の性能に何をもたらしますか？", keys=["学習", "未知"]),
    dict(group="para", q="LLMのハルシネーションはどのような現象ですか？", keys=["事実", "生成"]),
    dict(group="para", q="コンテキストウィンドウは何の上限を表しますか？", keys=["トークン", "上限"]),
    dict(group="outside", q="DPOとは何ですか？", keys=[]),
    dict(group="outside", q="BLEUスコアとは何ですか？", keys=[]),
    dict(group="outside", q="FlashAttentionとは何ですか？", keys=[]),
    dict(group="outside", q="RMSNormとは何ですか？", keys=[]),
    dict(group="outside", q="ゼロ知識証明とは何ですか？", keys=[]),
    dict(group="outside", q="2026年10月のGMOの不正アクセス事件の詳細を教えてください。", keys=[]),
]
assert len(EVAL_CASES) == 18
assert all(e['q'] in questions for e in EVAL_CASES if e['group']=='same')
assert all(e['q'] not in questions for e in EVAL_CASES if e['group']!='same')
print({group: sum(e['group']==group for e in EVAL_CASES) for group in ('same','para','outside')})
(WORK/'eval_questions.json').write_text(json.dumps(EVAL_CASES,ensure_ascii=False,indent=2),encoding='utf-8')


{'same': 6, 'para': 6, 'outside': 6}


1765

## 3. モデルの読み込みと、学習前評価
Qwen3.5-0.8Bを32bitで読み込む。既知・言い換え・用語集外の18問の生成を保存する。

In [6]:
import torch, time, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
pd.set_option('display.max_colwidth', 150)
assert torch.cuda.is_available(), 'T4 GPUを接続してください'
DEVICE = 'cuda'
tok = AutoTokenizer.from_pretrained(CONFIG['base_model'], revision=CONFIG['revision'])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG['base_model'], revision=CONFIG['revision'], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint()/1e9, 2)
print('model', CONFIG['base_model'], '| GPU:',torch.cuda.get_device_name(0),'| model GB:',BASE_MEMORY_GB)

def generate(m, question: str):
    # 教師データと同じ「user→assistant」会話形式を使用（system命令は追加しない）。
    messages=[{'role':'user','content':question}]
    prompt=tok.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
    inputs=tok(prompt,return_tensors='pt').to(m.device)
    with torch.no_grad():
        out=m.generate(**inputs,max_new_tokens=CONFIG['max_new_tokens'],pad_token_id=tok.pad_token_id,**CONFIG['gen'])
    return tok.decode(out[0][inputs['input_ids'].shape[1]:],skip_special_tokens=True).strip()

def evaluation(m, label):
    rows=[]; t0=time.perf_counter()
    for i,case in enumerate(EVAL_CASES,1):
        raw=generate(m,case['q']); group=case['group']
        if group!='outside':
            passed=all(word.casefold() in raw.casefold() for word in case['keys'])
        else:
            passed=None # 外部の出来事や講座外の知識は、自動採点で真偽を断定しない
        acknowledges_absence=any(w in raw for w in ('用語集に記載','用語集にない','用語集にはない','確認が必要','わかりません','分かりません','知らない','情報がありません'))
        rows.append({'id':i,'group':group,'question':case['q'],'must_include':case['keys'],
                     'answer':raw,'keyword_ok':passed,'absence_acknowledged':acknowledges_absence})
    summary={'label':label,'n':len(rows),'seconds':round(time.perf_counter()-t0,1)}
    for group in ('same','para'):
        subset=[r for r in rows if r['group']==group]
        summary[group+'_keyword_rate']=round(sum(r['keyword_ok'] for r in subset)/len(subset),3)
    subset=[r for r in rows if r['group']=='outside']
    summary['outside_absence_acknowledged_rate']=round(sum(r['absence_acknowledged'] for r in subset)/len(subset),3)
    (WORK/f'eval_{label}.json').write_text(json.dumps({'summary':summary,'rows':rows},ensure_ascii=False,indent=2),encoding='utf-8')
    print(json.dumps(summary,ensure_ascii=False))
    display(pd.DataFrame(rows)[['group','question','answer','keyword_ok','absence_acknowledged']])
    return summary, rows
before_summary,before_rows=evaluation(model,'before')


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

model Qwen/Qwen3.5-0.8B | GPU: Tesla T4 | model GB: 3.01


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{"label": "before", "n": 18, "seconds": 212.8, "same_keyword_rate": 0.167, "para_keyword_rate": 0.333, "outside_absence_acknowledged_rate": 0.0}


,group,question,answer,keyword_ok,absence_acknowledged
0,same,「トークン」とは何ですか？,**トークン（Token）** は、インターネット上で情報やデータを送信する際に使用される**「単位」**または**「通貨」**のことです。\n\n一般的に、以下の 3 つの用途で使われます：\n\n1. **Web 通信（Web 3.0）**\n * 仮想通貨（例：Bitcoi...,True,False
1,same,「LoRA」とは何ですか？,**LoRA** は、大規模言語モデル（LLM）のトレーニングにおいて非常に重要な技術の一つです。\n\n簡単に言うと、**「ローカルなモデル（小さなモデル）を、大規模なモデル（大きなモデル）に追加して、より高い精度や表現力を得るための手法」**です。\n\n具体的には、以下の 3 つのステ...,False,False
2,same,「量子化」とは何ですか？,**「量子化」**とは、物理学において**「量子力学」**という理論の核心を捉える概念です。\n\n簡単に言うと、**「連続的なものではなく、特定の「量子」の単位でしか変化しない性質」**を指します。\n\nこの概念は、1900 年、ニュートンやプランクによって発見された**「プランク定数」...,False,False
3,same,「GitHub Codespaces」とは何ですか？,**GitHub Codespaces** は、GitHub のプラットフォーム上で、**コードをリアルタイムで実行する環境**を提供するサービスです。\n\n簡単に言うと、「コードを書く」「コードを動かす」ための、GitHub 上で手軽に構築できる**仮想環境**です。\n\n主な特徴と使...,False,False
4,same,「過学習」とは何ですか？,**過学習**（Overfitting）とは、機械学習や統計学習において、モデルが学習したデータに**過剰に適合**し、**真のデータに欠ける**現象を指します。\n\n簡単に言うと、「学習したことが全部で、真の答えも全部で」という状態です。\n\n具体的には以下の 3 つの要素が重なって起...,False,False
5,same,「MCP（Model Context Protocol）」とは何ですか？,**MCP（Model Context Protocol）** は、大規模言語モデル（LLM）や AI 技術において、**「モデルの出力を人間が理解しやすく、かつ、ユーザーの意図を正確に伝達できる仕組み」**を指します。\n\n従来の AI 技術では、モデルが出力した文書やコードは、人間がそ...,False,False
6,para,LoRAでは、どの重みを固定し、何を学習するのですか？,LoRA（Low-Rank Adaptation）は、大規模言語モデル（LLM）の学習において非常に効果的な手法の一つです。その核心は、**「固定された重み（固定パラメータ）」と「学習する重み（学習パラメータ）」の分離**にあります。\n\n具体的には以下のようになります。\n\n### 1...,False,False
7,para,モデルの重みを4ビットにしてメモリを節約する技術は？,モデルの重みを 4 ビット（16 進数）にすると、メモリを大幅に節約できます。これは主に **メモリ効率化**（Memory Efficiency）という技術の核心です。\n\n具体的には、以下の 3 つの観点から考えられます。\n\n### 1. 計算量とメモリ量の対比（計算効率化）\n4...,False,False
8,para,MCPはAIアプリと外部ツールの連携にどう役立ちますか？,MCP（Model Context Protocol）は、AI アプリケーションと外部ツール（API、データベース、画像生成など）の連携を**「単一の API 呼び出し」で統一する**という概念です。\n\n従来の「個別の API を一つずつ使ったり、個別のモデルを個別に使う」スタイルと、M...,False,False
9,para,過学習は未知の入力の性能に何をもたらしますか？,過学習（Overfitting）は、**未知の入力の性能に悪影響**をもたらします。\n\nなぜそのような現象が発生するか、そしてそのメカニズムは以下の通りです。\n\n1. **学習データの過剰依存**: 過学習は、学習データ（訓練データ）に過度に集中することで発生します。学習モデルは、...,True,False


## 4. LoRAで用語集116件を学習
標準演習とは異なり、`glossary_sft.jsonl` の **`messages` を直接 `Dataset.from_list`** で渡す。学習後はadapterを別ファイルで保存する。

In [7]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer,SFTConfig
train_ds=Dataset.from_list(qa)
model.config.use_cache=False
lora=LoraConfig(r=CONFIG['lora_r'],lora_alpha=CONFIG['lora_alpha'],lora_dropout=0.05,bias='none',
                task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
sft=SFTConfig(output_dir=str(WORK/'ckpt'),num_train_epochs=CONFIG['num_train_epochs'],
              per_device_train_batch_size=4,gradient_accumulation_steps=2,
              learning_rate=CONFIG['lr'],lr_scheduler_type='cosine',warmup_steps=CONFIG['warmup_steps'],
              logging_steps=5,save_steps=15,save_total_limit=2,fp16=False,max_length=CONFIG['max_seq_len'],
              packing=False,report_to='none',seed=CONFIG['seed'])
trainer=SFTTrainer(model=model,args=sft,train_dataset=train_ds,peft_config=lora,processing_class=tok)
trainer.model.print_trainable_parameters()
t0=time.perf_counter()
trainer.train()
train_seconds=round(time.perf_counter()-t0,1)
tuned=trainer.model
tuned.config.use_cache=True
_=tuned.eval()
ADAPTER_DIR=WORK/'adapter';tuned.save_pretrained(ADAPTER_DIR)
print('学習時間（秒）',train_seconds,'| steps',trainer.state.global_step,'| final_loss',trainer.state.log_history[-1])


Tokenizing train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
5,3.706408
10,2.844986
15,2.505803
20,1.942597
25,1.794491
30,1.721200
35,1.271426
40,1.322993
45,1.215941


学習時間（秒） 118.6 | steps 45 | final_loss {'train_runtime': 118.2183, 'train_samples_per_second': 2.944, 'train_steps_per_second': 0.381, 'total_flos': 97179405904896.0, 'train_loss': 2.0362048149108887, 'epoch': 3.0, 'step': 45}


## 5. 学習後の同一条件での評価
学習前後の生成条件と18問は完全に同じ。モデルに差し込んだadapterの効果を比較する。

In [8]:
after_summary,after_rows=evaluation(tuned,'after')
compare=pd.DataFrame([before_summary,after_summary]).set_index('label')
display(compare)
detail=[]
for b,a in zip(before_rows,after_rows):
    detail.append({'種別':b['group'],'質問':b['question'],'学習前':b['answer'],'学習後':a['answer'],
                   '前・語句':b['keyword_ok'],'後・語句':a['keyword_ok'],
                   '前・未掲載を認めた':b['absence_acknowledged'],'後・未掲載を認めた':a['absence_acknowledged']})
display(pd.DataFrame(detail))
print('注意：用語集外の設問にはモデルが既存知識で正しく答える場合もある。',
      '自動検査は正誤の判定ではなく、「未掲載と明示したか」の記録である。')


{"label": "after", "n": 18, "seconds": 63.3, "same_keyword_rate": 0.667, "para_keyword_rate": 0.333, "outside_absence_acknowledged_rate": 0.0}


,group,question,answer,keyword_ok,absence_acknowledged
0,same,「トークン」とは何ですか？,LLMがテキストを扱う最小単位。1つの単語、記号、ハイパーフラグ、スペースなどによって数値が異なる。APIのレート制限やコスト計算の基礎となる。,True,False
1,same,「LoRA」とは何ですか？,学習データでモデルを学習させるのに使う小さなパラメータの更新手法。学習率と更新回数で調整できる。,False,False
2,same,「量子化」とは何ですか？,モデルの重みをより小さなビット数で表し、メモリ使用量や計算量を増やす手法。精度は少し落ちるが、学習するデータや出力の形式に依存する。,True,False
3,same,「GitHub Codespaces」とは何ですか？,GitHubが提供するクラウド上の開発環境。静的コードチェック、CI/CD、ツールチェーン、コードレビュー、Gitアカウントの共有・使用・削除ができる。無料枠は一定時間あたりのリソースとチェックイン数がある。,True,False
4,same,「過学習」とは何ですか？,学習データに過ぎないデータでモデルが過度に適合し、未知のデータで性能が落ちる状態。重要な入力や出力、評価指標に依存しすぎたり、評価指標を過信したりする。,True,False
5,same,「MCP（Model Context Protocol）」とは何ですか？,エージェントのツールを公開するオープンなプロトコル。公開したツールは、他のエージェントが受け取る入力・出力の型（ツール呼び出しの形式）を、MCPサーバーに渡す。公開しないツールは、エージェントに渡す入力として受け入れない。公開したツールは、サーバーに接続し、呼び出しをMCPクライアントから受...,False,False
6,para,LoRAでは、どの重みを固定し、何を学習するのですか？,学習する重みをLoRAの学習重み（低ランク重み）と、固定する重みをLoRAの固定重み（高ランク重み）の2つに分ける。学習重みは2次元のベクトル（低ランク重みは2次元、高ランク重みは1次元）で表し、固定重みは1次元のベクトル。,False,False
7,para,モデルの重みを4ビットにしてメモリを節約する技術は？,1ビットのメモリを節約するために、2つの値を1ビットで表す。0と1を同じビットに丸める。,False,False
8,para,MCPはAIアプリと外部ツールの連携にどう役立ちますか？,ツールを公開するAPIのURLをMCPサーバーに書き込み、他のAIエージェントがそれをMCPサーバーとして認識してツールを呼ぶ仕組み。公開するAPIのURLをMCPサーバーに書き込むと、他のエージェントがそれをMCPサーバーとして認識してツールを呼ぶ。公開するAPIのURLをMCPサーバーに...,False,False
9,para,過学習は未知の入力の性能に何をもたらしますか？,未知の入力に使うモデルの性能を、学習したモデルの性能に比べて大きく上回る性能を期待して使う入力に使うモデル。学習したモデルは、学習した入力に使うモデルの性能に比べて大きく下回る性能を期待して使う入力に使うモデル。,True,False


,n,seconds,same_keyword_rate,para_keyword_rate,outside_absence_acknowledged_rate
label,,,,,
before,18,212.8,0.167,0.333,0.0
after,18,63.3,0.667,0.333,0.0


,種別,質問,学習前,学習後,前・語句,後・語句,前・未掲載を認めた,後・未掲載を認めた
0,same,「トークン」とは何ですか？,**トークン（Token）** は、インターネット上で情報やデータを送信する際に使用される**「単位」**または**「通貨」**のことです。\n\n一般的に、以下の 3 つの用途で使われます：\n\n1. **Web 通信（Web 3.0）**\n * 仮想通貨（例：Bitcoi...,LLMがテキストを扱う最小単位。1つの単語、記号、ハイパーフラグ、スペースなどによって数値が異なる。APIのレート制限やコスト計算の基礎となる。,True,True,False,False
1,same,「LoRA」とは何ですか？,**LoRA** は、大規模言語モデル（LLM）のトレーニングにおいて非常に重要な技術の一つです。\n\n簡単に言うと、**「ローカルなモデル（小さなモデル）を、大規模なモデル（大きなモデル）に追加して、より高い精度や表現力を得るための手法」**です。\n\n具体的には、以下の 3 つのステ...,学習データでモデルを学習させるのに使う小さなパラメータの更新手法。学習率と更新回数で調整できる。,False,False,False,False
2,same,「量子化」とは何ですか？,**「量子化」**とは、物理学において**「量子力学」**という理論の核心を捉える概念です。\n\n簡単に言うと、**「連続的なものではなく、特定の「量子」の単位でしか変化しない性質」**を指します。\n\nこの概念は、1900 年、ニュートンやプランクによって発見された**「プランク定数」...,モデルの重みをより小さなビット数で表し、メモリ使用量や計算量を増やす手法。精度は少し落ちるが、学習するデータや出力の形式に依存する。,False,True,False,False
3,same,「GitHub Codespaces」とは何ですか？,**GitHub Codespaces** は、GitHub のプラットフォーム上で、**コードをリアルタイムで実行する環境**を提供するサービスです。\n\n簡単に言うと、「コードを書く」「コードを動かす」ための、GitHub 上で手軽に構築できる**仮想環境**です。\n\n主な特徴と使...,GitHubが提供するクラウド上の開発環境。静的コードチェック、CI/CD、ツールチェーン、コードレビュー、Gitアカウントの共有・使用・削除ができる。無料枠は一定時間あたりのリソースとチェックイン数がある。,False,True,False,False
4,same,「過学習」とは何ですか？,**過学習**（Overfitting）とは、機械学習や統計学習において、モデルが学習したデータに**過剰に適合**し、**真のデータに欠ける**現象を指します。\n\n簡単に言うと、「学習したことが全部で、真の答えも全部で」という状態です。\n\n具体的には以下の 3 つの要素が重なって起...,学習データに過ぎないデータでモデルが過度に適合し、未知のデータで性能が落ちる状態。重要な入力や出力、評価指標に依存しすぎたり、評価指標を過信したりする。,False,True,False,False
5,same,「MCP（Model Context Protocol）」とは何ですか？,**MCP（Model Context Protocol）** は、大規模言語モデル（LLM）や AI 技術において、**「モデルの出力を人間が理解しやすく、かつ、ユーザーの意図を正確に伝達できる仕組み」**を指します。\n\n従来の AI 技術では、モデルが出力した文書やコードは、人間がそ...,エージェントのツールを公開するオープンなプロトコル。公開したツールは、他のエージェントが受け取る入力・出力の型（ツール呼び出しの形式）を、MCPサーバーに渡す。公開しないツールは、エージェントに渡す入力として受け入れない。公開したツールは、サーバーに接続し、呼び出しをMCPクライアントから受...,False,False,False,False
6,para,LoRAでは、どの重みを固定し、何を学習するのですか？,LoRA（Low-Rank Adaptation）は、大規模言語モデル（LLM）の学習において非常に効果的な手法の一つです。その核心は、**「固定された重み（固定パラメータ）」と「学習する重み（学習パラメータ）」の分離**にあります。\n\n具体的には以下のようになります。\n\n### 1...,学習する重みをLoRAの学習重み（低ランク重み）と、固定する重みをLoRAの固定重み（高ランク重み）の2つに分ける。学習重みは2次元のベクトル（低ランク重みは2次元、高ランク重みは1次元）で表し、固定重みは1次元のベクトル。,False,False,False,False
7,para,モデルの重みを4ビットにしてメモリを節約する技術は？,モデルの重みを 4 ビット（16 進数）にすると、メモリを大幅に節約できます。これは主に **メモリ効率化**（Memory Efficiency）という技術の核心です。\n\n具体的には、以下の 3 つの観点から考えられます。\n\n### 1. 計算量とメモリ量の対比（計算効率化）\n4...,1ビットのメモリを節約するために、2つの値を1ビットで表す。0と1を同じビットに丸める。,False,False,False,False
8,para,MCPはAIアプリと外部ツールの連携にどう役立ちますか？,MCP（Model Context Protocol）は、AI アプリケーションと外部ツール（API、データベース、画像生成など）の連携を**「単一の API 呼び出し」で統一する**という概念です。\n\n従来の「個別の API を一つずつ使ったり、個別のモデルを個別に使う」スタイルと、M...,ツールを公開するAPIのURLをMCPサーバーに書き込み、他のAIエージェントがそれをMCPサーバーとして認識してツールを呼ぶ仕組み。公開するAPIのURLをMCPサーバーに書き込むと、他のエージェントがそれをMCPサーバーとして認識してツールを呼ぶ。公開するAPIのURLをMCPサーバーに...,False,False,False,False
9,para,過学習は未知の入力の性能に何をもたらしますか？,過学習（Overfitting）は、**未知の入力の性能に悪影響**をもたらします。\n\nなぜそのような現象が発生するか、そしてそのメカニズムは以下の通りです。\n\n1. **学習データの過剰依存**: 過学習は、学習データ（訓練データ）に過度に集中することで発生します。学習モデルは、...,未知の入力に使うモデルの性能を、学習したモデルの性能に比べて大きく上回る性能を期待して使う入力に使うモデル。学習したモデルは、学習した入力に使うモデルの性能に比べて大きく下回る性能を期待して使う入力に使うモデル。,True,True,False,False


注意：用語集外の設問にはモデルが既存知識で正しく答える場合もある。 自動検査は正誤の判定ではなく、「未掲載と明示したか」の記録である。


## 6. 簡易用語集検索と比較（モデル学習なし）
講義の要求に従って、*用語集を検索して答える仕組み* を作る。比較対象は**文字列一致の簡易検索**であり、埋め込み検索や本格的RAGではない。言い換えで検索失敗しやすい点も観察する。

In [10]:
import re
# 116件のuser質問から「用語」を抽出。2つの言い回しは同一定義なので、最初の定義を採用。
glossary={}
for row in qa:
    q=row['messages'][0]['content']; ans=row['messages'][1]['content']
    mt=re.search(r'「(.+?)」とは',q)
    if mt: glossary[mt.group(1)]=ans

# 単純検索。用語の長い順にチェックし、例えば QLoRA を LoRA より優先する。
# よく使う略称だけを明示しておく（未知の問いをモデルの推測で埋めない）。
aliases={
    'MCP':'MCP（Model Context Protocol）',
    'LLM':'LLM（大規模言語モデル）',
    'SFT':'教師ありファインチューニング（SFT）',
}
def glossary_lookup(q):
    for term in sorted(glossary, key=len,reverse=True):
        if term.casefold() in q.casefold(): return glossary[term],term
    for alias,term in aliases.items():
        if re.search(r'(?<![A-Za-z])'+re.escape(alias)+r'(?![A-Za-z])',q,re.I) and term in glossary:
            return glossary[term],term
    return '用語集に該当項目が見つかりません。確認が必要です。',None
search_rows=[]
for case in EVAL_CASES:
    ans,term=glossary_lookup(case['q'])
    if case['group']=='outside':
        ok=(term is None) # 用語集の未掲載を検出できたか
    else:
        ok=all(word.casefold() in ans.casefold() for word in case['keys'])
    search_rows.append({'group':case['group'],'question':case['q'],'hit':term,'answer':ans,'score':ok})
display(pd.DataFrame(search_rows)[['group','question','hit','score','answer']])
print('簡易検索の群別スコア:', {g:round(sum(r['score'] for r in search_rows if r['group']==g)/sum(r['group']==g for r in search_rows),3) for g in ['same','para','outside']})


,group,question,hit,score,answer
0,same,「トークン」とは何ですか？,トークン,True,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。
1,same,「LoRA」とは何ですか？,LoRA,True,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。
2,same,「量子化」とは何ですか？,量子化,True,モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。
3,same,「GitHub Codespaces」とは何ですか？,GitHub Codespaces,True,GitHubが提供するクラウド上の開発環境。ブラウザからVS Codeを開き、repoに書かれた設定どおりの環境をすぐに使える。個人アカウントには毎月の無料枠があり、使わないときは停止しないと枠を消費し続ける。
4,same,「過学習」とは何ですか？,過学習,True,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。
5,same,「MCP（Model Context Protocol）」とは何ですか？,MCP（Model Context Protocol）,True,AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。
6,para,LoRAでは、どの重みを固定し、何を学習するのですか？,LoRA,True,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。
7,para,モデルの重みを4ビットにしてメモリを節約する技術は？,None,False,用語集に該当項目が見つかりません。確認が必要です。
8,para,MCPはAIアプリと外部ツールの連携にどう役立ちますか？,外部ツール,True,Webページの取得、ファイル操作、データベース、他社のAPIなど、エージェントのプログラムの外にある機能をツールとして使うこと。第3回の関数ツールは同じプロセスの中で動くが、外部ツールはプロセスや機械をまたいで呼び出す。MCPはそのための共通の規約。
9,para,過学習は未知の入力の性能に何をもたらしますか？,過学習,True,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。


簡易検索の群別スコア: {'same': 1.0, 'para': 0.833, 'outside': 1.0}


## 7. 結果の保存と提出
`/content/lec02_advanced/` の `result.json` `eval_before.json` `eval_after.json` `eval_questions.json` `retrieval_results.json` を提出資料としてダウンロード。`lectures/02/answer.md` の「宿題」欄にレベル・変更点・比較数値・FTと検索のメリット/デメリットを記録する。**実行データにない数値を報告しない。**

In [11]:
# trainer.state.log_historyの末尾はtrain_runtime等の場合があるので、直近lossを抽出
losses=[x['loss'] for x in trainer.state.log_history if 'loss' in x]
result={
  'created_jst': __import__('datetime').datetime.now(__import__('zoneinfo').ZoneInfo('Asia/Tokyo')).strftime('%Y/%m/%d %H:%M'),
  'level':'発展','dataset':'glossary_sft.jsonl','dataset_count':len(qa),
  'model':CONFIG['base_model'],'revision':CONFIG['revision'],'epochs':CONFIG['num_train_epochs'],
  'learning_rate':CONFIG['lr'],'lora_r':CONFIG['lora_r'],'lora_alpha':CONFIG['lora_alpha'],
  'generation':CONFIG['gen'],'test_count':len(EVAL_CASES),'train_seconds':train_seconds,
  'steps':trainer.state.global_step,'last_logged_loss':losses[-1] if losses else None,
  'before':before_summary,'after':after_summary,
  'retrieval_scores': {g:round(sum(r['score'] for r in search_rows if r['group']==g)/sum(r['group']==g for r in search_rows),3) for g in ['same','para','outside']},
  'executed_training_myself':True,
  'evaluation_limitations':'自動採点は必須語句一致のみで意味上の正確性を保証しない。用語集外は未掲載を自発的に認めた率であり、知識の真偽は別途原文確認が必要。簡易検索は文字列一致。'
}
(WORK/'retrieval_results.json').write_text(json.dumps(search_rows,ensure_ascii=False,indent=2),encoding='utf-8')
(WORK/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(result,ensure_ascii=False,indent=2))
# 5つの成果ファイルをZIP化してまとめてダウンロード（元データ116件はCodespaces側で保持）
import zipfile
archive=WORK/'lec02_advanced_results.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for name in ('result.json','eval_before.json','eval_after.json','eval_questions.json','retrieval_results.json'):
        z.write(WORK/name,arcname=name)
print('保存しました:',archive)
from google.colab import files
files.download(str(archive))


{
  "created_jst": "2026/10/28 12:48",
  "level": "発展",
  "dataset": "glossary_sft.jsonl",
  "dataset_count": 116,
  "model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "epochs": 3,
  "learning_rate": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "generation": {
    "do_sample": false
  },
  "test_count": 18,
  "train_seconds": 118.6,
  "steps": 45,
  "last_logged_loss": 1.2159406661987304,
  "before": {
    "label": "before",
    "n": 18,
    "seconds": 212.8,
    "same_keyword_rate": 0.167,
    "para_keyword_rate": 0.333,
    "outside_absence_acknowledged_rate": 0.0
  },
  "after": {
    "label": "after",
    "n": 18,
    "seconds": 63.3,
    "same_keyword_rate": 0.667,
    "para_keyword_rate": 0.333,
    "outside_absence_acknowledged_rate": 0.0
  },
  "retrieval_scores": {
    "same": 1.0,
    "para": 0.833,
    "outside": 1.0
  },
  "executed_training_myself": true,
  "evaluation_limitations": "自動採点は必須語句一致のみで意味上の正確性を保証しない。用語集外は未掲載を自発的に認めた率であり、知

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>